# USGS earthquakes: inspect depth before adding a third dimension

How do map, time, cross-section and scene views change the reading of a selected earthquake catalog? We use 78 recorded events from January 2024 near Japan and make coordinate and depth conventions explicit.

## Objectives

- Validate event coordinates, UTC times and magnitude types
- Construct map, timeline, depth and 3D views from one selection
- Explain projection approximation, depth sign and vertical exaggeration

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import repo_examples as rx

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("usgs"))

In [ ]:
events = rx.read_csv("usgs/events.csv")
raw = rx.read_json("usgs/events.geojson")
events["time_utc"] = pd.to_datetime(events.time_utc, utc=True)
coords = np.asarray([f["geometry"]["coordinates"] for f in raw["features"]])
assert len(events) == 78 and events.id.is_unique
assert np.allclose(coords, events[["longitude", "latitude", "depth_km"]])
assert events.time_utc.min() >= pd.Timestamp("2024-01-01", tz="UTC")
assert events.time_utc.max() < pd.Timestamp("2024-02-01", tz="UTC")
display(events[["id", "time_utc", "magnitude", "magnitude_type", "depth_km", "place"]].head())
display(events.magnitude_type.value_counts().rename("events"))
# Fixed local equirectangular teaching approximation, not a geodetic solution.
origin_lon, origin_lat = 138.0, 38.0
east_km = np.radians(events.longitude-origin_lon) * 6371 * np.cos(np.radians(origin_lat))
north_km = np.radians(events.latitude-origin_lat) * 6371
z_km = -events.depth_km.to_numpy()  # positive depth becomes negative z
size = 12 + 18*(events.magnitude-4.5+1)**2  # legibility rule, not rupture/energy area

## View 1: location and time share an event selection

Color encodes catalog depth, positive downward. Marker area is a stated legibility rule using magnitude; it does not scale earthquake energy or rupture area. Magnitude is logarithmic and reported types differ. The timeline reveals temporal clustering that the map alone cannot show.

**Method connection:** the catalog's Cartesian coordinates (§2), Scatterplot (§9) and Timeline (§12) answer complementary questions. Epicenter maps, cross-sections and the 3D scene extend those constructions to spatial coordinates. View selection and a consistent legend matter more than adding perspective. None of these patterns predicts a later earthquake.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
points = axes[0].scatter(events.longitude, events.latitude, s=size, c=events.depth_km,
                         cmap="viridis_r", alpha=.75, edgecolor="white", linewidth=.4)
axes[0].set_aspect(1/np.cos(np.radians(origin_lat)))
rx.finish(axes[0], "Selected epicenters — no basemap", "Longitude (° east)", "Latitude (° north)")
axes[1].scatter(events.time_utc, events.magnitude, s=30, c=events.depth_km, cmap="viridis_r")
rx.finish(axes[1], "UTC time and reported magnitude", "January 2024 (UTC)", "Magnitude (mixed reported types)")
axes[1].tick_params(axis="x", labelrotation=35)
fig.colorbar(points, ax=axes, label="Catalog depth (km, positive down)", shrink=.8)
plt.show()

## View 2: use a cross-section before perspective

In this section, y is catalog depth with the axis inverted, so deeper events are physically lower on the page. The horizontal axis is approximate east displacement from 138° E at 38° N; north displacement is compressed out. This is not a fault-plane model or a tomographic section.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(east_km, events.depth_km, s=size/2, c=events.magnitude, cmap="magma")
axes[0].invert_yaxis()
rx.finish(axes[0], "East–depth projection", "Approximate east displacement (km)", "Depth (km, positive down)")
axes[1].hist(events.depth_km, bins=[0, 20, 40, 80, 160, 320, 640], color="#317a94", edgecolor="white")
rx.finish(axes[1], "Depth counts in unequal-width bins", "Catalog depth (km)", "Selected events per bin (not density)")
fig.tight_layout()
plt.show()

## View 3: disclose the scene transform

The scene uses z = −depth, so deeper events have a smaller vertical coordinate. All three numeric axes are kilometers, but the display below applies a selectable vertical exaggeration. Changing it moves no catalog event; it changes perception of vertical separation. The origin and approximate projection are visible in code. An interactive local [earthquake story](https://jltobias.github.io/JupyterLite-Data-Visualization/repository-examples/) offers 2D, 3D, table and downloads without requiring a GPU or external map service.

In [ ]:
vertical_exaggeration = 2.0
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection="3d")
cloud = ax.scatter(east_km, north_km, z_km*vertical_exaggeration, c=events.depth_km,
                   cmap="viridis_r", s=size/3, alpha=.8)
ax.set(xlabel="Approximate east (km)", ylabel="Approximate north (km)",
       zlabel=f"Display z = −depth × {vertical_exaggeration:g} (km)",
       title=f"78 catalog hypocenters; vertical exaggeration {vertical_exaggeration:g}×")
spans = [np.ptp(east_km), np.ptp(north_km), np.ptp(z_km)*vertical_exaggeration]
ax.set_box_aspect(spans)
fig.colorbar(cloud, ax=ax, label="Unscaled catalog depth (km)", shrink=.6)
plt.show()
assert np.allclose(z_km, -coords[:, 2])
display(pd.DataFrame({"origin_lon_deg": [origin_lon], "origin_lat_deg": [origin_lat],
                      "projection": ["local equirectangular approximation"], "vertical_exaggeration": [vertical_exaggeration]}))

## Exercise

Set vertical exaggeration to 1 and 5. Compare the apparent clusters with the unchanged cross-section and numeric table. Restrict to magnitude ≥5 and report the changed count and selection. Describe what additional catalog-completeness and tectonic evidence is needed before interpreting a spatial cluster.

## Limitations and reflection

The query selects magnitude ≥4.5, 30–46° N, 128–148° E, January 2024 UTC; it is not all seismicity or an aftershock classification. Events may be revised; retrieval and update times are retained. Location/depth uncertainty is not encoded because this tidy selection does not supply it. Magnitude types differ. A local planar approximation over this large region has distortion; use an appropriate projected/geodesic workflow externally for precise distances. Perspective, overlap and exaggerated depth can distort pattern perception.

## Citations and repository connections

[Fixed USGS event query](https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2024-01-01&endtime=2024-02-01&minmagnitude=4.5&minlatitude=30&maxlatitude=46&minlongitude=128&maxlongitude=148&orderby=time-asc), [USGS GeoJSON specification](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php), [copyright and credit](https://www.usgs.gov/information-policies-and-instructions/copyrights-and-credits). Scene/narrative pattern connection: [Sea-Level-Rise 6076bea2c878](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/tree/6076bea2c8781285b204a5af0c3cd0ccc5d5ef8e); this course's earthquake scene is original and uses actual USGS event records.

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.